# 3.2. Object-Oriented Design for Implementation
D2L의 Object-Oriented Design for Implementation 장을 PyTorch 기준으로 정리함.

이 장은 선형회귀 수식을 배우지 않고 앞으로 딥러닝 모델 구현할 때 사용할 코드 구조 설계하는 장이다.
딥러닝에서는 데이터, 모델, 손실 함수, 최적화 알고리즘, 학습 루프 같은 구성요소가 반복적으로 등장한다

D2L에서는 이 구성요소들을 객체로 나눠 관리한다.

**1. Module**
- 모델, loss, optimizer 관련 기능을 담당한다.

**2. DataModule**
- 학습 데이터와 검증 데이터를 제공한다.

**3. Trainer**
- Module과 DataModule을 받아서 실제 학습 루프를 실행한다.

앞으로 구조는 대략 이런식이다.

data = DataModule(...)
model = Module(...)
trainer = Trainer(...)
trainer.fit(model, data)

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 왜 객체지향 구조를 쓰나?

나쁜 방식은 모든 코드를 한곳에 섞어쓰는 것이다.

```py
# 나쁜 구조 예시
X, y = load_data()
w = torch.randn(...)
b = torch.zeros(...)

for epoch in range(num_epochs):
    y_hat = X @ w + b
    loss = ...
    loss.backward()
    optimizer.step()
```

처음엔 이게 쉬워보이지만 모델이 바뀌고, 데이터가 바뀌고, optimizer가 바뀌면 코드 전체를 계속 뜯어고쳐야 한다.  
좋은 구조는 역할을 나눈다.

```py
data = MyDataModule()
model = MyModel()
trainer = Trainer(max_epochs=10)

trainer.fit(model, data)
```

이렇게 하면 모델만 바꾸고 싶을 때는 model만 바꾸면 된다. 데이터만 바꾸고 싶을 때는 data만 바꾸면 된다.  
D2L에서도 이 구조가 프로젝트 간 컴포넌트 재사용을 쉽게 만든다고 설명한다.

## 2. Utilities

D2L은 Jupyter Notebook에서 객체지향 코드를 쓰기 쉽게 하려고 몇 가지 도구를 만들었다.  
일반적인 파이썬 라이브러리처럼 긴 클래스를 한번에 정의하면 읽기 어렵다. 그래서 클래스를 여러 코드 블록으로 나눠 유틸리티를 만든다.

### 2.1 add_to_class

D2L의 add_to_class는 이미 만들어진 클래스에 나중에 메서드를 추가하는 함수이다.

In [2]:
def add_to_class(Class):
    
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

In [3]:
class A: # 예를 들어서 이런 클래스가 있을때
    def __init__(self):
        self.b = 1

a = A()

# class A: # 원래라면 do() 메서드는 클래스 A 안에 정의되어야 한다.
#     def __init__(self):
#         self.b = 1

#     def do(self):
#         print(self.b)


@add_to_class(A) # D2L은 노트북 설명을 위해 이렇게 썼다
def do(self):
    print('Class attribute "b" is', self.b)

a.do()

Class attribute "b" is 1


### 2.2 HyperParameters

D2L의 HyperParameters는 __init__에 들어온 인자들을 자동으로 self.~~~형태로 저장해준다.  
원문에서는 save_hyperparameters를 호출하면 a, b 같은 생성자 인자가 객체 속성으로 저장되고,  
ignore=['c']처럼 일부 인자는 저장하지 않을 수 있음을 보여준다.

In [ ]:
# class B: # 일반적으로 이렇게 정의할 수 있다.
#     def __init__(self, a, b, c):
#         self.a = a
#         self.b = b
#         self.c = c

class B(d2l.HyperParameters): # D2L에서는 이걸 이런식으로 줄인다.
    def __init__(self, a, b, c):
        self.save_hyperparameters(ignore=['c'])

# self.a = a # 내부적으로 이런식으로 일어난다고 보면 된다.
# self.b = b
# self.c는 ignore 이니 저장 X

### 2.3 ProgressBoard

ProgressBoard는 학습 중 loss 같은 값을 그래프로 그리기 위한 도구다.   
D2L은 TensorBoard보다 단순한 시각화 도구로 ProgressBoard를 사용한다고 설명한다.   
draw(x, y, label, every_n)은 특정 점을 그래프에 찍고, every_n은 너무 많은 점을 다 그리지 않도록 일부를 평균 내어 부드럽게 보여주는 역할을 한다.

In [ ]:
board = d2l.ProgressBoard('x')

for x in np.arange(0, 10, 0.1):
    board.draw(x, np.sin(x), 'sin', every_n=2)
    board.draw(x, np.cos(x), 'cos', every_n=10)

## 3. Models: Module

D2L의 Module은 앞으로 만들 모든 모델의 기본 클래스다. PyTorch 버전에서는 nn.Module을 상속한다. Module에는 최소한 세 가지 메서드가 필요하다.   
__init__은 학습 가능한 파라미터를 저장하고, training_step은 데이터 배치를 받아 loss를 계산하고, configure_optimizers는 파라미터를 업데이트할 optimizer를 반환한다. 선택적으로 validation_step도 정의할 수 있다.

In [ ]:
class Module(nn.Module): # 단순화 하면 이런식의 구조이다.
    def __init__(self):
        super().__init__()

    def loss(self, y_hat, y): # 예측값 y_hat과 정답 y를 비교해서 loss를 계산한다.
        raise NotImplementedError

    def forward(self, X): # 입력 X를 받아 예측 y_hat을 만든다.
        return self.net(X) # 실제 신경망 구조이다. 선형회귀는 nn.Linear같은 게 들어간다.

    def training_step(self, batch): # 배치 하나를 받아 forward와 loss계산을 한다.
        X = batch[:-1] # batch[:-1]가 튜플 형태로 들고 올 수 있기 때문에 (X, y)형태라 (X,)가 된다.
        y = batch[-1] 
        y_hat = self(*X) # *는 튜플을 풀어 함수 인자로 넘긴다
        l = self.loss(y_hat, y)
        return l

    def validation_step(self, batch):
        X = batch[:-1]
        y = batch[-1]
        y_hat = self(*X)
        l = self.loss(y_hat, y)
        return l

    def configure_optimizers(self): # SGD, Adam 같은 optimizer를 만든다.
        raise NotImplementedError # 이건 구체적으로 구현하지 않는다는 의미이다. Module은 뼈대만 제공하기 때문에 loss나 optimizer는 직접 구현해야한다.

In [ ]:
class LinearRegression(Module): # 선형 회귀모델 예시
    def __init__(self, lr):
        super().__init__()
        self.net = nn.Linear(2, 1) 
        self.lr = lr

    def loss(self, y_hat, y):
        return ((y_hat - y) ** 2).mean()

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=self.lr)

## 4. Data: DataModule

DataModule은 데이터 담당 클래스다. D2L은 DataModule을 데이터의 기본 클래스로 두고, 보통 __init__에서 데이터를 준비한다고 설명한다.  
필요하면 다운로드나 전처리도 여기서 한다. train_dataloader는 학습 데이터 로더를 반환하고, val_dataloader는 검증 데이터 로더를 반환한다.  
데이터 로더는 사용할 때마다 batch를 하나씩 내보내는 Python generator처럼 동작한다.

In [ ]:
class DataModule(d2l.HyperParameters): # DataModule은 데이터를 준비하고 batch 단위로 꺼내주는 역할을 한다.
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train): # train이 True면 학습 데이터, False면 검증 데이터
        raise NotImplementedError # 데이터셋마다 로딩 방식이 다르기 때문에 자식 클래스에서 구현해야함.

    def train_dataloader(self): # 학습용 batch를 제공
        return self.get_dataloader(train=True)

    def val_dataloader(self): # 검증용 batch를 제공
        return self.get_dataloader(train=False)

## 5. Training: Trainer

Trainer는 실제 학습을 실행하는 클래스다. D2L은 Trainer가 Module 안의 학습 가능한 파라미터를 DataModule이 제공한 데이터로 학습한다고 설명한다.  
핵심 메서드는 fit(model, data)이고, 이것은 전체 데이터셋을 max_epochs번 반복하면서 모델을 학습한다.

In [ ]:
class Trainer(d2l.HyperParameters):
    def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
        self.save_hyperparameters()
        assert num_gpus == 0, 'No GPU support yet'

    def prepare_data(self, data): # 1. data에서 train_dataloader와 val_dataloader를 가져온다.
        self.train_dataloader = data.train_dataloader()
        self.val_dataloader = data.val_dataloader()
        self.num_train_batches = len(self.train_dataloader)
        self.num_val_batches = (
            len(self.val_dataloader)
            if self.val_dataloader is not None
            else 0
        )

    def prepare_model(self, model): # 2. trainer가 model을 관리할 수 있게 연결한다.
        model.trainer = self
        self.model = model

    def fit(self, model, data):
        self.prepare_data(data)
        self.prepare_model(model)
        self.optim = model.configure_optimizers() # 3. 모델이 사용할 optimizer를 가져온다.

        self.epoch = 0
        self.train_batch_idx = 0
        self.val_batch_idx = 0

        for self.epoch in range(self.max_epochs): # 4. max_epochs만큼 fit_epoch()를 반복한다.
            self.fit_epoch()

    def fit_epoch(self):
        raise NotImplementedError

## 6. 전체 구조 한번에 보기

**DataModule**
- 데이터를 준비한다.
- batch를 제공한다.

**Module**
- 모델 구조를 가진다.
- forward를 수행한다.
- loss를 계산한다.
- optimizer를 만든다.

**Trainer**
- DataModule에서 batch를 가져온다.
- Module에 batch를 넣어 loss를 계산한다.
- backward를 수행한다.
- optimizer.step()으로 파라미터를 업데이트한다.

## 7. 정리

- 이 장은 선형회귀 수식이 아니라 딥러닝 구현 구조를 설계하는 장이다.
- 딥러닝 학습에는 데이터, 모델, loss, optimizer, training loop가 반복해서 등장한다.
- D2L은 이 구성요소를 Module, DataModule, Trainer로 나눈다.
- Module은 모델, loss, optimizer를 담당한다.
- DataModule은 학습 데이터와 검증 데이터를 batch 단위로 제공한다.
- Trainer는 모델과 데이터를 받아 실제 학습 루프를 실행한다.
- nn.Module에서는 forward를 정의하면 model(X)처럼 호출할 수 있다.
- training_step은 batch 하나를 받아 예측값과 loss를 계산한다.
- configure_optimizers는 optimizer를 반환한다.
- get_dataloader와 fit_epoch는 아직 구체적으로 구현하지 않고, 나중 장에서 자식 클래스나 추가 구현으로 채운다.
- add_to_class, HyperParameters, ProgressBoard는 D2L이 노트북 설명을 편하게 하기 위해 만든 도구다.
- 지금은 D2L 내부 유틸리티보다 Module/DataModule/Trainer의 역할 분리를 이해하는 것이 중요하다.